# Phase 5 — Research Chat UI

Demonstrates the conversational `ResearchChat` over the local corpus via Claude tool use.

This notebook uses a **fully mocked Anthropic client**, so it runs without an API key — ideal for CI and quick reproductions.

In [1]:
from pathlib import Path
from tempfile import mkdtemp
from unittest.mock import MagicMock

from ah_research.chat.chat import ResearchChat
from ah_research.chat.session import ChatSession
from ah_research.chat.tools import TOOLS, ChatDeps

## The 8 tools

`ResearchChat` exposes these tools to Claude via the Anthropic tool-use API:

In [2]:
[t["name"] for t in TOOLS]

['list_universe',
 'get_corpus_summary',
 'get_screener_row',
 'get_dossier',
 'get_profile_markdown',
 'get_graded_profile',
 'search_filings',
 'construct_portfolio']

## Build a mocked Anthropic client

We canned two response rounds: round 1 issues a `tool_use` for `list_universe`; round 2 returns the final text.

In [3]:
def _mock_text_block(text):
    b = MagicMock()
    b.type = "text"
    b.text = text
    return b


def _mock_tool_use_block(name, input_, tool_use_id):
    b = MagicMock()
    b.type = "tool_use"
    b.name = name
    b.input = input_
    b.id = tool_use_id
    return b


def _mock_response(blocks):
    r = MagicMock()
    r.content = blocks
    return r


client = MagicMock()
client.messages.create.side_effect = [
    _mock_response([_mock_tool_use_block("list_universe", {}, tool_use_id="tu_1")]),
    _mock_response(
        [_mock_text_block("I found 2 tickers in the local corpus: 600519.SH and 000001.SZ.")]
    ),
]

## Build mock repositories and a session

In [4]:
filings_repo = MagicMock()
filings_repo.list_symbols.return_value = ["600519.SH", "000001.SZ"]
profile_repo = MagicMock()
profile_repo.list_symbols.return_value = ["600519.SH"]

root = Path(mkdtemp(prefix="phase5-chat-"))
session = ChatSession.new(anchor="600519.SH", model="claude-sonnet-4-6", root=root)
deps = ChatDeps(
    data_repo=MagicMock(),
    filings_repo=filings_repo,
    profile_repo=profile_repo,
    profile_grader=None,
)
chat = ResearchChat(session=session, deps=deps, client=client, max_iterations=5)

## Run one turn

In [5]:
answer = chat.send("How many symbols are in the corpus?")
print(answer)

I found 2 tickers in the local corpus: 600519.SH and 000001.SZ.


## Inspect the persisted session

Every turn — user, tool_result, and assistant — is appended to the JSONL session file atomically so `--resume` works even if the API crashes mid-turn.

In [6]:
for t in chat.session.turns:
    print(f"{t.role:12} | tool={t.tool_name or '-':20} | content={t.content[:80]!r}")

user         | tool=-                    | content='How many symbols are in the corpus?'
tool_result  | tool=list_universe        | content='{"symbols": ["000001.SZ", "600519.SH"], "n_with_profile": 1, "n_with_filings": 2'
assistant    | tool=-                    | content='I found 2 tickers in the local corpus: 600519.SH and 000001.SZ.'


In [7]:
print("Session file:", session.path)
print(session.path.read_text())

Session file: /var/folders/1w/s14mdy7s1s57q_ftj0hglmjm0000gn/T/phase5-chat-dyeh089x/20260430T175757-600519-sh.jsonl
{"_meta": true, "session_id": "20260430T175757-600519-sh", "anchor_symbol": "600519.SH", "model": "claude-sonnet-4-6"}
{"role": "user", "content": "How many symbols are in the corpus?", "timestamp": "2026-04-30T17:57:57.045767+00:00"}
{"role": "tool_result", "content": "{\"symbols\": [\"000001.SZ\", \"600519.SH\"], \"n_with_profile\": 1, \"n_with_filings\": 2}", "tool_name": "list_universe", "tool_use_id": "tu_1", "timestamp": "2026-04-30T17:57:57.045981+00:00"}
{"role": "assistant", "content": "I found 2 tickers in the local corpus: 600519.SH and 000001.SZ.", "timestamp": "2026-04-30T17:57:57.046080+00:00"}



## Architecture recap

- `ChatSession` — JSONL-persisted history at `~/.ah-research/chat/<session-id>.jsonl`.
- `ChatDeps` — bundles `DataRepository`, `FilingsRepository`, `ProfileRepository`, and optional `ProfileGrader`.
- `TOOLS` — 8 tool schemas exposed to Claude (list_universe, get_corpus_summary, get_screener_row, get_dossier, get_profile_markdown, get_graded_profile, search_filings, construct_portfolio).
- `ResearchChat.send(user_text)` — runs the tool-use loop: system prompt → user → tool_use → tool_result → final text.
- CLI entry point: `ah chat [TICKER] [--resume ID] [--list]`.